# 02 - Baseline covariates

Derives baseline comorbidities (10-year look-back) and the most recent laboratory/measurement values (eGFR, BMI and others) at DOAC initiation, and records which continuous values are missing. Missing eGFR and BMI are multiply imputed later, inside each hazard-ratio notebook (miceforest).

**Manuscript items:** Covariates (Methods); Table S5; input to Table 1, Table S8, Figs S2-S3

**Inputs:** `DERIVED_DIR/cohort_DDI_major.xlsx`, CPRD Observation extract, CPRD lookups, comorbidity / bleeding / hospitalisation code lists

**Outputs:** `DERIVED_DIR/pats_comorbidities_DDI.xlsx`, `DERIVED_DIR/pats_with_missing_comorbidities_values_df.xlsx` (patient-level - never share)

*Derived from the original analysis notebook `2_comorbidities_DDI.ipynb`. Outputs have been cleared; no patient-level data are included.*

In [ ]:
# Project file locations are defined once in paths.py (same folder as this notebook).
# Edit DATA_DIR / OUTPUT_DIR there, or set the DDI_DATA_DIR / DDI_OUTPUT_DIR environment variables.
import os
from paths import *

In [ ]:
import time
import matplotlib.pyplot as plt
import os
import glob
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import multiprocessing as mp
import zipfile
from multiprocessing import Pool, cpu_count


In [ ]:
n_processes = 100  # Number of processes to use for parallel processing

In [ ]:
doac_exposure = pd.read_excel(os.path.join(DERIVED_DIR, "cohort_DDI_major.xlsx"), dtype=str)
doac_exposure["study_start_date"] = pd.to_datetime(doac_exposure["study_start_date"], errors="coerce")
doac_exposure["study_end_date"] = pd.to_datetime(doac_exposure["study_end_date"], errors="coerce")

doac_exposure["age_at_study_start"] = pd.to_numeric(doac_exposure["age_at_study_start"], errors="coerce")
doac_exposure["taken_warfarin_1w_before"] = pd.to_numeric(doac_exposure["taken_warfarin_1w_before"], errors="coerce")
doac_exposure["atleast_1y_med_hist_flag"] = pd.to_numeric(doac_exposure["atleast_1y_med_hist_flag"], errors="coerce")

doac_exposure = doac_exposure[(doac_exposure["age_at_study_start"]>=18) &
                               (doac_exposure["taken_warfarin_1w_before"]==0) &
                              (doac_exposure["atleast_1y_med_hist_flag"]==1) &
                 (doac_exposure["taken_warfarin_during_study"] =="0") ]


doac_exposure.drop(columns=["taken_warfarin_1w_before", "atleast_1y_med_hist_flag", "taken_warfarin_during_study", 
                            "taken_warfarin_1m_before"], inplace=True)
doac_exposure

In [ ]:
pats_comorbidities = doac_exposure[["patid", "study_start_date", "age_at_study_start"]].copy()

In [ ]:
prod_dict = pd.read_csv(PRODUCT_DICT, sep="\t" ,dtype=str)

emis_cat = pd.read_csv(os.path.join(LOOKUP_DIR, "EmisCodeCat.txt"), sep="\t", dtype=str)
obstypeid = pd.read_csv(os.path.join(LOOKUP_DIR, "ObsType.txt"), sep="\t", dtype=str)
Numunit = pd.read_csv(os.path.join(LOOKUP_DIR, "NumUnit.txt"), sep="\t", dtype=str)
gender = pd.read_csv(os.path.join(LOOKUP_DIR, "Gender.txt"), sep="\t", dtype=str)

med_dict = pd.read_csv(os.path.join(LOOKUP_DIR, "med_dict.txt"), sep="\t", dtype=str)
#med_dict = med_dict[["MedCodeId", "Observations", "Term", "SnomedCTConceptId", "EmisCodeCategoryId"]]
med_dict["emiscodecat"] = med_dict["EmisCodeCategoryId"].map(emis_cat.set_index("emiscodecatid")["Description"])
med_dict.drop(columns=["EmisCodeCategoryId"], inplace=True)

icd_codes = pd.read_csv(os.path.join(LOOKUP_DIR, "snomed_icd10_mappings.txt"), sep="\t", dtype=str)
med_dict["icd10_code"] = med_dict["SnomedCTConceptId"].map(icd_codes.set_index("SNOMED")["ICD10_CODE"])

#### Identify Comorbidities and Extract Laboratory Test Values

First, we extract binary indicators for the following comorbidities for each patient in the cohort:

1. Hypertension  
2. Stroke  
3. Diabetes  
4. Cancer  
5. Smoking  
6. Coronary Heart Disease (CHD)  
7. Alcohol problem  
8. Major bleeding history  
9. Liver disease  
10. Atrial Fibrillation
11. Venous Thromboembolism

Next, we extract the following laboratory test values:

10. eGFR  
11. BMI  
12. Haemoglobin  
13. Total cholesterol  
14. HDL cholesterol  
15. LDL cholesterol  
16. Platelet count
17. Systolib blood pressure
18. Diastolic blood pressure
19. Pulse rate

##### Step 5.1

In [ ]:
hypertension_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "comorbidities_codes.xlsx"), sheet_name="hypertension", dtype=str)
stroke_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "comorbidities_codes.xlsx"), sheet_name="stroke_tia_se", dtype=str)
diabetes_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "comorbidities_codes.xlsx"), sheet_name="diabetes", dtype=str)
cancer_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "comorbidities_codes.xlsx"), sheet_name="cancer", dtype=str)
smoking_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "comorbidities_codes.xlsx"), sheet_name="smoking", dtype=str)
chd_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "comorbidities_codes.xlsx"), sheet_name="chd", dtype=str) # Coronary Heart Disease
alcohol_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "comorbidities_codes.xlsx"), sheet_name="alcohol", dtype=str)
anemia_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "comorbidities_codes.xlsx"), sheet_name="anemia", dtype=str)
heart_failure_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "comorbidities_codes.xlsx"), sheet_name="heart_failure", dtype=str)
angina_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "comorbidities_codes.xlsx"), sheet_name="angina", dtype=str)
myocardial_infraction_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "comorbidities_codes.xlsx"), sheet_name="myocardial_infarction", dtype=str)
af_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "comorbidities_codes.xlsx"), sheet_name="atrial_fibrillation", dtype=str)
vte_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "comorbidities_codes.xlsx"), sheet_name="vte", dtype=str)


In [ ]:
comorbidities_codes = {
    "hypertension": set(hypertension_codes["MedCodeId"].tolist()),
    "stroke":       set(stroke_codes["MedCodeId"].tolist()),
    "diabetes":     set(diabetes_codes["MedCodeId"].tolist()),
    "cancer":       set(cancer_codes["MedCodeId"].tolist()),
    "smoking":      set(smoking_codes["MedCodeId"].tolist()),
    "chd":          set(chd_codes["MedCodeId"].tolist()),
    "alcohol":      set(alcohol_codes["MedCodeId"].tolist()),
    "anemia":       set(anemia_codes["MedCodeId"].tolist()),
    "heart_failure": set(heart_failure_codes["MedCodeId"].tolist()),
    "angina":       set(angina_codes["MedCodeId"].tolist()),
    "myocardial_infraction": set(myocardial_infraction_codes["MedCodeId"].tolist()),
    "atrial_fibrillation": set(af_codes["MedCodeId"].tolist()),
    "venous_thromboembolism": set(vte_codes["MedCodeId"].tolist())
}

doac_lookup = pats_comorbidities.set_index("patid")["study_start_date"]
valid_patids = set(pats_comorbidities["patid"])

def process_file(file_name):
    with zipfile.ZipFile(OBSERVATION_ZIP, 'r') as zip_ref:
        df = pd.read_csv(zip_ref.open(file_name), sep="\t", dtype=str)

    df = df[df["patid"].isin(valid_patids)].copy()
    if df.empty:
        return {key: set() for key in comorbidities_codes}
    

    df["study_start_date"] = df["patid"].map(doac_lookup)
    df["obsdate"]        = pd.to_datetime(df["obsdate"], errors="coerce")
    df["study_start_date"] = pd.to_datetime(df["study_start_date"], errors="coerce")

    df = df.dropna(subset=["obsdate", "study_start_date"])

    # Keep only records within 10 years before the study start date to limit the data we need to process
    df = df[(df["obsdate"] <= df["study_start_date"]) & (df["obsdate"] >= df["study_start_date"] - pd.Timedelta(days=3650))]

    comorbidity_flags = {}
    for comorbidity, codes in comorbidities_codes.items():
        # Boolean mask — no Python loop over rows
        mask = df["medcodeid"].isin(codes)
        comorbidity_flags[comorbidity] = set(df.loc[mask, "patid"])

    return comorbidity_flags


if __name__ == "__main__":
    with zipfile.ZipFile(OBSERVATION_ZIP, 'r') as zip_ref:
        txt_files = [f for f in zip_ref.namelist() if f.endswith(".txt")]

    with Pool(n_processes) as pool:
        results = pool.map(process_file, txt_files)

    combined_flags = {key: set() for key in comorbidities_codes}
    for result in results:
        for comorbidity, patids in result.items():
            combined_flags[comorbidity].update(patids)

In [ ]:
##add comorbidities flags to pats_comorbidities dataframe
for comorbidity in comorbidities_codes.keys():
    pats_comorbidities[comorbidity] = pats_comorbidities["patid"].apply(lambda x: 1 if x in combined_flags[comorbidity] else 0)

##### Step 5.2
Major bleeding history

In [ ]:
bleeding_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "bleeding_codes.xlsx"),dtype=str)
hospitalisation_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "hospitalisation_codes.xlsx"), dtype=str)

In [ ]:
# Prepare list of batches to consider
batches_to_consider = []
with zipfile.ZipFile(OBSERVATION_ZIP, 'r') as zip_ref:
    for batch in zip_ref.namelist():
        batches_to_consider.append(batch)

def process_file(file_name):
    
    with zipfile.ZipFile(OBSERVATION_ZIP, 'r') as zip_ref:
        df = pd.read_csv(zip_ref.open(file_name), sep="\t", dtype=str)
    
    df = df[df["patid"].isin(doac_exposure["patid"])]
    df["obsdate"] = pd.to_datetime(df["obsdate"])
    df["study_start_date"] = df["patid"].map(doac_exposure.set_index("patid")["study_start_date"])
    df = df[(df["obsdate"] <= df["study_start_date"]) & (df["obsdate"] >= df["study_start_date"] - pd.Timedelta(days=3650))]
    bleeding_events = df[df["medcodeid"].isin(bleeding_codes["MedCodeId"])][["patid", "obsdate", "medcodeid"]]
    hospital_events = df[df["medcodeid"].isin(hospitalisation_codes["MedCodeId"])][["patid", "obsdate"]]
    
    return bleeding_events, hospital_events

if __name__ == "__main__":
    # Use a reasonable number of processes
    #n_processes = min(100, cpu_count())

    with Pool(processes=n_processes) as pool:
        results = pool.map(process_file, batches_to_consider)

    # Combine results
    bleeding_events_hist= pd.concat([res[0] for res in results], ignore_index=True)
    hospital_events_hist = pd.concat([res[1] for res in results], ignore_index=True)

In [ ]:
bleeding_hospitalizations_hist = pd.merge(hospital_events_hist, bleeding_events_hist, on="patid", how="inner", suffixes=('_hospital', '_bleeding'))
bleeding_hospitalizations_hist = bleeding_hospitalizations_hist[(bleeding_hospitalizations_hist["obsdate_hospital"]>=bleeding_hospitalizations_hist["obsdate_bleeding"]) &
                        (bleeding_hospitalizations_hist["obsdate_hospital"]<= bleeding_hospitalizations_hist["obsdate_bleeding"] + pd.Timedelta(days=7))]
bleeding_hospitalizations_hist.sort_values(by=["patid", "obsdate_bleeding", "obsdate_hospital"], inplace=True)
bleeding_hospitalizations_hist = bleeding_hospitalizations_hist.groupby(["patid", "obsdate_bleeding"]).first().reset_index()

bleeding_hospitalizations_hist.sort_values(by=["patid", "obsdate_bleeding"], inplace=True)
bleeding_hospitalizations_hist.drop_duplicates(subset=["patid"], inplace=True)

In [ ]:
pats_comorbidities["major_bled_hist"] = pats_comorbidities["patid"].isin(bleeding_hospitalizations_hist["patid"])
pats_comorbidities["major_bled_hist"] = pats_comorbidities["major_bled_hist"].astype(int)

print(f"Number of patients with major bleeding events before DOAC start: {pats_comorbidities['major_bled_hist'].sum()}")

##### Step 5.3
Liver disease

In [ ]:
liver_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "comorbidities_codes.xlsx"), sheet_name="liver",dtype=str)
kidney_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "comorbidities_codes.xlsx"),sheet_name="ckd", dtype=str)
bmi_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "comorbidities_codes.xlsx"),sheet_name="bmi",dtype=str)
hemoglobin_codes= pd.read_excel(os.path.join(CODE_LIST_DIR, "comorbidities_codes.xlsx"),sheet_name="hemoglobin",dtype=str)
platelet_count_code = pd.read_excel(os.path.join(CODE_LIST_DIR, "comorbidities_codes.xlsx"),sheet_name="platelet_count",dtype=str)
cholestrol_total_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "comorbidities_codes.xlsx"),sheet_name="cholestrol_total",dtype=str)
cholestrol_ldl_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "comorbidities_codes.xlsx"),sheet_name="cholestrol_ldl",dtype=str)
cholestrol_hdl_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "comorbidities_codes.xlsx"),sheet_name="cholestrol_hdl",dtype=str)
systolic_bp_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "comorbidities_codes.xlsx"),sheet_name="systolic_bp",dtype=str)
diastolic_bp_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "comorbidities_codes.xlsx"),sheet_name="diastolic_bp",dtype=str)
pulse_rate_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "comorbidities_codes.xlsx"),sheet_name="pulse_rate",dtype=str)

In [ ]:
batches_to_consider = []
with zipfile.ZipFile(OBSERVATION_ZIP, 'r') as zip_ref:
    for batch in zip_ref.namelist():
        batches_to_consider.append(batch)

def process_file(file_name):
    
    with zipfile.ZipFile(OBSERVATION_ZIP, 'r') as zip_ref:
        df = pd.read_csv(zip_ref.open(file_name), sep="\t", dtype=str)
    
    df = df[df["patid"].isin(doac_exposure["patid"])]

    df["obsdate"] = pd.to_datetime(df["obsdate"])
    df["study_start_date"] = df["patid"].map(doac_exposure.set_index("patid")["study_start_date"])
    df = df[(df["obsdate"] <= df["study_start_date"]) & (df["obsdate"] >= df["study_start_date"] - pd.Timedelta(days=3650))]

    liver_data = df[df["medcodeid"].isin(liver_codes["MedCodeId"])]
    kidney_data = df[df["medcodeid"].isin(kidney_codes["MedCodeId"])]
    bmi_data = df[df["medcodeid"].isin(bmi_codes["MedCodeId"])]
    hemoglobin_data = df[df["medcodeid"].isin(hemoglobin_codes["MedCodeId"])]
    platelet_data = df[df["medcodeid"].isin(platelet_count_code["MedCodeId"])]
    cholestrol_data = df[df["medcodeid"].isin(cholestrol_total_codes["MedCodeId"].to_list() +
                                               cholestrol_ldl_codes["MedCodeId"].to_list() +
                                                 cholestrol_hdl_codes["MedCodeId"].to_list())]
    systolic_bp_data = df[df["medcodeid"].isin(systolic_bp_codes["MedCodeId"])]
    diastolic_bp_data = df[df["medcodeid"].isin(diastolic_bp_codes["MedCodeId"])]
    pulse_rate_data = df[df["medcodeid"].isin(pulse_rate_codes["MedCodeId"])]


    return liver_data, kidney_data, bmi_data, hemoglobin_data, platelet_data, cholestrol_data, systolic_bp_data, diastolic_bp_data, pulse_rate_data

if __name__ == "__main__":
    with Pool(processes=n_processes) as pool:
        results = pool.map(process_file, batches_to_consider)

    liver_data = pd.concat([result[0] for result in results], ignore_index=True)
    kidney_data = pd.concat([result[1] for result in results], ignore_index=True)
    bmi_data = pd.concat([result[2] for result in results], ignore_index=True)
    hemoglobin_data = pd.concat([result[3] for result in results], ignore_index=True)
    platelet_data = pd.concat([result[4] for result in results], ignore_index=True)
    cholestrol_data = pd.concat([result[5] for result in results], ignore_index=True)
    systolic_bp_data = pd.concat([result[6] for result in results], ignore_index=True)
    diastolic_bp_data = pd.concat([result[7] for result in results], ignore_index=True)
    pulse_rate_data = pd.concat([result[8] for result in results], ignore_index=True)

In [ ]:
#🔹 ALT (normal: 7–56 U/L)
#ULN = 56
#3× ULN = 168 U/L

#🔹 AST (normal: 10–40 U/L)
#ULN = 40
#3× ULN = 120 U/L

#🔹 ALP (normal: 45–120 U/L)
#ULN = 120
#3× ULN = 360 U/L

liver_data["Term"] = liver_data["medcodeid"].map(med_dict.set_index("MedCodeId")["Term"])

liver_measurement_alt = liver_data[(liver_data["Term"].str.contains("ALT", case=False, na=False)) & (liver_data["value"].astype(float) > 168)]
liver_measurement_alt.sort_values(by=["patid", "obsdate"], inplace=True)
liver_measurement_alt = liver_measurement_alt.groupby("patid").tail(1)

liver_measurement_ast =  liver_data[(liver_data["Term"].str.contains("AST", case=False, na=False)) & (liver_data["value"].astype(float) > 120)]
liver_measurement_ast.sort_values(by=["patid", "obsdate"], inplace=True)
liver_measurement_ast = liver_measurement_ast.groupby("patid").tail(1)
liver_measurement = pd.concat([liver_measurement_alt, liver_measurement_ast])
liver_measurement.sort_values(by=["patid", "obsdate"], inplace=True)
liver_cat = liver_data[~liver_data["Term"].str.contains("ALT|AST", case=False, na=False)]
all_liver_impairment = pd.concat([liver_measurement, liver_cat])
##remove duplicates
all_liver_impairment.sort_values(by=["patid", "obsdate"], inplace=True)
all_liver_impairment = all_liver_impairment.groupby("patid").tail(1)

pats_comorbidities["liver_disease"] = pats_comorbidities["patid"].isin(all_liver_impairment["patid"]) | pats_comorbidities["patid"].isin(liver_cat["patid"])
pats_comorbidities["liver_disease"] = pats_comorbidities["liver_disease"].astype(int)

##### Step 5.4
eGFR

In [ ]:
### Kidney function ####

kidney_data["Term"] = kidney_data["medcodeid"].map(med_dict.set_index("MedCodeId")["Term"])

kidney_data["obsdate"] = pd.to_datetime(kidney_data["obsdate"])
kidney_data["study_start_date"] = kidney_data["patid"].map(doac_exposure.set_index("patid")["study_start_date"])
kidney_data["study_start_date"] = pd.to_datetime(kidney_data["study_start_date"])
kidney_data = kidney_data[kidney_data["obsdate"] <= kidney_data["study_start_date"]]
kidney_data["obstypeid"] = kidney_data["obstypeid"].map(obstypeid.set_index("obstypeid")["Description"])
kidney_data["numunitid"] = kidney_data["numunitid"].map(Numunit.set_index("numunitid")["Description"])
#kidney = kidney[kidney["obsdate"]>= kidney["start_date"] - pd.Timedelta(days=365)]

kidney_measurement_terms = ["GFR","Blood urea/renal function", "Renal profile", "Renal function tests", "glomerular filtration rate",
                            "Measurement of renal function"]

kidney_measurement = kidney_data[kidney_data["Term"].str.contains("|".join(kidney_measurement_terms), case=False, na=False)]
kidney_measurement["value"] = pd.to_numeric(kidney_measurement["value"], errors='coerce')
kidney_measurement = kidney_measurement[(kidney_measurement["value"] > 0) & (kidney_measurement["value"] < 200)]
#show latest kidney function test
kidney_measurement.sort_values(by=["patid", "obsdate"], inplace=True)
kidney_measurement_latest = kidney_measurement.groupby("patid").tail(1) # get latest record for each patient
kidney_measurement_latest = kidney_measurement_latest[["patid", "Term", "value", "numunitid", "obsdate"]]

print(f"Number of patients with kidney function measurement: {kidney_measurement_latest['patid'].nunique()}")

In [ ]:
total_pats = pats_comorbidities["patid"].nunique()
pats_with_missing_kidney = set(pats_comorbidities["patid"]) - set(kidney_measurement_latest["patid"])

print(f"Number of patients with missing kidney function measurement: {len(pats_with_missing_kidney)} ({len(pats_with_missing_kidney)/total_pats*100:.2f}%)")


In [ ]:
pats_comorbidities["ckd_value"] = pats_comorbidities["patid"].map(kidney_measurement_latest.set_index("patid")["value"])
pats_comorbidities["ckd_value"] = pd.to_numeric(pats_comorbidities["ckd_value"], errors='coerce')



##### Step 5.5
BMI

In [ ]:
bmi_data["study_start_date"] = bmi_data["patid"].map(doac_exposure.set_index("patid")["study_start_date"])
bmi_data["study_start_date"] = pd.to_datetime(bmi_data["study_start_date"])
bmi_data["obsdate"] = pd.to_datetime(bmi_data["obsdate"])
bmi_data["value"] = pd.to_numeric(bmi_data["value"], errors='coerce')


bmi_data = bmi_data[(bmi_data["value"] > 10) & (bmi_data["value"] < 80)] # filter out implausible BMI values
#bmi = bmi[bmi["obsdate"]>= bmi["start_date"] - pd.Timedelta(days=365)]
bmi_data.sort_values(by=["patid", "obsdate"], inplace=True)
bmi_latest = bmi_data.groupby("patid").tail(1) # get latest record for each patient
bmi_latest = bmi_latest[["patid", "value", "obsdate", "study_start_date"]]

In [ ]:

pats_comorbidities["bmi_value"] = pats_comorbidities["patid"].map(bmi_latest.set_index("patid")["value"])
pats_comorbidities["bmi_value"] = pd.to_numeric(pats_comorbidities["bmi_value"], errors='coerce')



##### Step 5.6
Haemoglobin Level

In [ ]:
hemoglobin_data["obstypeid"] = hemoglobin_data["obstypeid"].map(obstypeid.set_index("obstypeid")["Description"])
hemoglobin_data["numunitid"] = hemoglobin_data["numunitid"].map(Numunit.set_index("numunitid")["Description"])
hemoglobin_data["value"] = np.where(hemoglobin_data["numunitid"] == "g/L", hemoglobin_data["value"].astype(float) / 10, hemoglobin_data["value"].astype(float))
hemoglobin_data["value"] = pd.to_numeric(hemoglobin_data["value"], errors='coerce')
hemoglobin_data = hemoglobin_data[hemoglobin_data["numunitid"].isin(["g/L", "g/dL"])]
hemoglobin_data["numunitid"].replace({"g/L": "g/dL"}, inplace=True)
hemoglobin_data = hemoglobin_data[(hemoglobin_data["value"] >= 5) & (hemoglobin_data["value"] <= 20)] # filter out implausible hemoglobin values

##get latest hemoglobin measurement
hemoglobin_data.sort_values(by=["patid", "obsdate"], inplace=True)
hemoglobin_latest = hemoglobin_data.groupby("patid").tail(1) # get latest record for each patient

In [ ]:
pats_comorbidities["hemoglobin_value"] = pats_comorbidities["patid"].map(hemoglobin_latest.set_index("patid")["value"])
pats_comorbidities["hemoglobin_value"] = pd.to_numeric(pats_comorbidities["hemoglobin_value"], errors='coerce')



##### Step 5.7
Total Cholesterol, HDL Cholesterol, LDL Cholesterol

In [ ]:
cholestrol_data["term"] = cholestrol_data["medcodeid"].map(med_dict.set_index("MedCodeId")["Term"])
cholestrol_data["numunitid"] = cholestrol_data["numunitid"].map(Numunit.set_index("numunitid")["Description"])
cholestrol_data = cholestrol_data[cholestrol_data["value"].notna()]
cholestrol_data["obstypeid"] = cholestrol_data["obstypeid"].map(obstypeid.set_index("obstypeid")["Description"])
cholestrol_data

In [ ]:
cholestrol_data["term"] = cholestrol_data["medcodeid"].map(med_dict.set_index("MedCodeId")["Term"])
cholestrol_data["numunitid"] = cholestrol_data["numunitid"].map(Numunit.set_index("numunitid")["Description"])
cholestrol_data = cholestrol_data[(cholestrol_data["numunitid"] == "mmol/L") | (cholestrol_data["numunitid"].isna())]
cholestrol_data = cholestrol_data[cholestrol_data["value"].notna()]

cholestrol_total_data = cholestrol_data[cholestrol_data["medcodeid"].isin(cholestrol_total_codes["MedCodeId"])]
cholestrol_ldl_data = cholestrol_data[cholestrol_data["medcodeid"].isin(cholestrol_ldl_codes["MedCodeId"])]
cholestrol_hdl_data = cholestrol_data[cholestrol_data["medcodeid"].isin(cholestrol_hdl_codes["MedCodeId"])]

print("Total cholesterol records:", len(cholestrol_total_data["patid"].unique()))
print("LDL cholesterol records:", len(cholestrol_ldl_data["patid"].unique()))
print("HDL cholesterol records:", len(cholestrol_hdl_data["patid"].unique()))

In [ ]:
cholestrol_total_data["value"] = pd.to_numeric(cholestrol_total_data["value"], errors='coerce')
cholestrol_total_data = cholestrol_total_data[(cholestrol_total_data["value"] > 0) & (cholestrol_total_data["value"] < 15)]
cholestrol_total_data.sort_values(by=["patid", "obsdate"], inplace=True)
cholestrol_total_data = cholestrol_total_data.groupby("patid").tail(1) 


cholestrol_ldl_data["value"] = pd.to_numeric(cholestrol_ldl_data["value"], errors='coerce')
cholestrol_ldl_data = cholestrol_ldl_data[(cholestrol_ldl_data["value"] > 0) & (cholestrol_ldl_data["value"] < 15)]
cholestrol_ldl_data.sort_values(by=["patid", "obsdate"], inplace=True)
cholestrol_ldl_data = cholestrol_ldl_data.groupby("patid").tail(1)


cholestrol_hdl_data["value"] = pd.to_numeric(cholestrol_hdl_data["value"], errors='coerce')
cholestrol_hdl_data = cholestrol_hdl_data[(cholestrol_hdl_data["value"] > 0) & (cholestrol_hdl_data["value"] < 15)]
cholestrol_hdl_data.sort_values(by=["patid", "obsdate"], inplace=True)
cholestrol_hdl_data = cholestrol_hdl_data.groupby("patid").tail(1)


In [ ]:

pats_comorbidities["cholesterol_hdl"] = pats_comorbidities["patid"].map(cholestrol_hdl_data.set_index("patid")["value"])

########################### Total cholesterol ###########################

pats_comorbidities["cholesterol_total"] = pats_comorbidities["patid"].map(cholestrol_total_data.set_index("patid")["value"])
pats_comorbidities["cholesterol_total"] = pd.to_numeric(pats_comorbidities["cholesterol_total"], errors='coerce')


########################### HDL cholesterol ###########################
pats_comorbidities["cholesterol_hdl"] = pats_comorbidities["patid"].map(cholestrol_hdl_data.set_index("patid")["value"])
pats_comorbidities["cholesterol_hdl"] = pd.to_numeric(pats_comorbidities["cholesterol_hdl"], errors='coerce')


########################### LDL cholesterol ###########################
pats_comorbidities["cholesterol_ldl"] = pats_comorbidities["patid"].map(cholestrol_ldl_data.set_index("patid")["value"])
pats_comorbidities["cholesterol_ldl"] = pd.to_numeric(pats_comorbidities["cholesterol_ldl"], errors='coerce')



##### Step 5.8
Platelet Count

In [ ]:

platelet_data["value"] = pd.to_numeric(platelet_data["value"], errors='coerce')
platelet_data = platelet_data[platelet_data["value"].notna()]
platelet_data["value"] = platelet_data["value"].astype(float)
platelet_data.sort_values(by=["patid", "obsdate"], inplace=True)
platelet_latest = platelet_data.groupby("patid").tail(1)

##remove implausible values

platelet_latest = platelet_latest[(platelet_latest["value"] > 0) & (platelet_latest["value"] < 800)]

print(f"Number of patients with platelet count data: {len(platelet_latest)}")


In [ ]:
pats_comorbidities["platelet_count"] = pats_comorbidities["patid"].map(platelet_latest.set_index("patid")["value"])
pats_comorbidities["platelet_count"] = pd.to_numeric(pats_comorbidities["platelet_count"], errors='coerce')



##### Systolic Blood Pressure

In [ ]:
systolic_bp_data =systolic_bp_data[systolic_bp_data["patid"].isin(doac_exposure["patid"])]
systolic_bp_data.dropna(subset=["value"], inplace=True)
##get latest systolic blood pressure measurement
systolic_bp_data.sort_values(by=["patid", "obsdate"], inplace=True)
systolic_bp_latest = systolic_bp_data.groupby("patid").tail(1)
systolic_bp_latest = systolic_bp_latest[systolic_bp_latest["obsdate"]>= systolic_bp_latest["study_start_date"] - pd.Timedelta(days=365)]

systolic_bp_latest["value"] = pd.to_numeric(systolic_bp_latest["value"], errors='coerce')
systolic_bp_latest = systolic_bp_latest[(systolic_bp_latest["value"] > 50) & (systolic_bp_latest["value"] < 250)] # filter out implausible systolic blood pressure values

print(f"Number of patients with systolic blood pressure measurement in the year prior to DOAC start: {systolic_bp_latest['patid'].nunique()}")

In [ ]:
##impute missing systolic blood pressure values
pats_comorbidities["systolic_bp"] = pats_comorbidities["patid"].map(systolic_bp_latest.set_index("patid")["value"])
pats_comorbidities["systolic_bp"] = pd.to_numeric(pats_comorbidities["systolic_bp"], errors='coerce')



### Diastolic Blood Pressure

In [ ]:
diastolic_bp_data =diastolic_bp_data[diastolic_bp_data["patid"].isin(doac_exposure["patid"])]
diastolic_bp_data.dropna(subset=["value"], inplace=True)

##get latest diastolic blood pressure measurement
diastolic_bp_data.sort_values(by=["patid", "obsdate"], inplace=True)
diastolic_bp_latest = diastolic_bp_data.groupby("patid").tail(1)
diastolic_bp_latest = diastolic_bp_latest[diastolic_bp_latest["obsdate"]>= diastolic_bp_latest["study_start_date"] - pd.Timedelta(days=365)]

diastolic_bp_latest["value"] = pd.to_numeric(diastolic_bp_latest["value"], errors='coerce')
diastolic_bp_latest = diastolic_bp_latest[(diastolic_bp_latest["value"] > 30) & (diastolic_bp_latest["value"] < 150)] # filter out implausible diastolic blood pressure values


In [ ]:
pats_comorbidities["diastolic_bp"] = pats_comorbidities["patid"].map(diastolic_bp_latest.set_index("patid")["value"])
pats_comorbidities["diastolic_bp"] = pats_comorbidities["diastolic_bp"].astype(float)



### Pulse Rate

In [ ]:
pulse_rate_data = pulse_rate_data[pulse_rate_data["patid"].isin(doac_exposure["patid"])]
pulse_rate_data.dropna(subset=["value"], inplace=True)
##get latest pulse rate measurement
pulse_rate_data.sort_values(by=["patid", "obsdate"], inplace=True)
pulse_latest = pulse_rate_data.groupby("patid").tail(1)

pulse_latest["value"] = pd.to_numeric(pulse_latest["value"], errors='coerce')
pulse_latest = pulse_latest[(pulse_latest["value"] > 30) & (pulse_latest["value"] < 200)] # filter out implausible pulse rate values

In [ ]:
pats_comorbidities["pulse_rate"] = pats_comorbidities["patid"].map(pulse_latest.set_index("patid")["value"])
pats_comorbidities["pulse_rate"] = pats_comorbidities["pulse_rate"].astype(float)



In [ ]:
pats_comorbidities.to_excel(os.path.join(DERIVED_DIR, "pats_comorbidities_DDI.xlsx"), index=False)

In [ ]:
total_pats = pats_comorbidities["patid"].nunique()

pats_with_missing_ckd_value = len(list(set(pats_comorbidities["patid"]) - set(kidney_measurement_latest["patid"])))
pats_with_missing_bmi_value = len(list(set(pats_comorbidities["patid"]) - set(bmi_latest["patid"])))
pats_with_missing_hemoglobin_value = len(list(set(pats_comorbidities["patid"]) - set(hemoglobin_latest["patid"])))
pats_with_missing_cholesterol_total_value = len(list(set(pats_comorbidities["patid"]) - set(cholestrol_total_data["patid"])))
pats_with_missing_cholesterol_ldl_value = len(list(set(pats_comorbidities["patid"]) - set(cholestrol_ldl_data["patid"])))
pats_with_missing_cholesterol_hdl_value = len(list(set(pats_comorbidities["patid"]) - set(cholestrol_hdl_data["patid"])))
pats_with_missing_platelet_count_value = len(list(set(pats_comorbidities["patid"]) - set(platelet_latest["patid"])))
pats_with_missing_systolic_bp_value = len(list(set(pats_comorbidities["patid"]) - set(systolic_bp_latest["patid"])))
pats_with_missing_diastolic_bp_value = len(list(set(pats_comorbidities["patid"]) - set(diastolic_bp_latest["patid"])))
pats_with_missing_pulse_rate_value = len(list(set(pats_comorbidities["patid"]) - set(pulse_latest["patid"])))


print(f"Patients with missing CKD value: {pats_with_missing_ckd_value} ({pats_with_missing_ckd_value/total_pats*100:.2f}%)")
print(f"Patients with missing BMI value: {pats_with_missing_bmi_value} ({pats_with_missing_bmi_value/total_pats*100:.2f}%)")
print(f"Patients with missing Hemoglobin value: {pats_with_missing_hemoglobin_value} ({pats_with_missing_hemoglobin_value/total_pats*100:.2f}%)")
print(f"Patients with missing Cholesterol Total value: {pats_with_missing_cholesterol_total_value} ({pats_with_missing_cholesterol_total_value/total_pats*100:.2f}%)")
print(f"Patients with missing Cholesterol LDL value: {pats_with_missing_cholesterol_ldl_value} ({pats_with_missing_cholesterol_ldl_value/total_pats*100:.2f}%)")
print(f"Patients with missing Cholesterol HDL value: {pats_with_missing_cholesterol_hdl_value} ({pats_with_missing_cholesterol_hdl_value/total_pats*100:.2f}%)")
print(f"Patients with missing Platelet Count value: {pats_with_missing_platelet_count_value} ({pats_with_missing_platelet_count_value/total_pats*100:.2f}%)")
print(f"Patients with missing Systolic BP value: {pats_with_missing_systolic_bp_value} ({pats_with_missing_systolic_bp_value/total_pats*100:.2f}%)")
print(f"Patients with missing Diastolic BP value: {pats_with_missing_diastolic_bp_value} ({pats_with_missing_diastolic_bp_value/total_pats*100:.2f}%)")
print(f"Patients with missing Pulse Rate value: {pats_with_missing_pulse_rate_value} ({pats_with_missing_pulse_rate_value/total_pats*100:.2f}%)")


In [ ]:
pats_with_missing_values_df = pats_comorbidities[["patid"]]
pats_with_missing_values_df["ckd_value"] = pats_with_missing_values_df["patid"].isin(kidney_measurement_latest["patid"]).apply(lambda x: 0 if x else 1)
pats_with_missing_values_df["bmi_value"] = pats_with_missing_values_df["patid"].isin(bmi_latest["patid"]).apply(lambda x: 0 if x else 1)
pats_with_missing_values_df["hemoglobin_value"] = pats_with_missing_values_df["patid"].isin(hemoglobin_latest["patid"]).apply(lambda x: 0 if x else 1)
pats_with_missing_values_df["cholesterol_total"] = pats_with_missing_values_df["patid"].isin(cholestrol_total_data["patid"]).apply(lambda x: 0 if
    x else 1)
pats_with_missing_values_df["cholesterol_ldl"] = pats_with_missing_values_df["patid"].isin(cholestrol_ldl_data["patid"]).apply(lambda x: 0 if x else 1)
pats_with_missing_values_df["cholesterol_hdl"] = pats_with_missing_values_df["patid"].isin(cholestrol_hdl_data["patid"]).apply(lambda x: 0 if x else 1)
pats_with_missing_values_df["platelet_count"] = pats_with_missing_values_df["patid"].isin(platelet_latest["patid"]).apply(lambda x: 0 if x else 1)
pats_with_missing_values_df["systolic_bp"] = pats_with_missing_values_df["patid"].isin(systolic_bp_latest["patid"]).apply(lambda x: 0 if x else 1)
pats_with_missing_values_df["diastolic_bp"] = pats_with_missing_values_df["patid"].isin(diastolic_bp_latest["patid"]).apply(lambda x: 0 if x else 1)
pats_with_missing_values_df["pulse_rate"] = pats_with_missing_values_df["patid"].isin(pulse_latest["patid"]).apply(lambda x: 0 if x else 1)



In [ ]:
pats_with_missing_values_df.to_excel(os.path.join(DERIVED_DIR, "pats_with_missing_comorbidities_values_df.xlsx"), index=False)